# 01 — Exploration des données (EDA)

**Projet** : Moteur de recommandation de catalogue e-commerce avec scikit-learn
**Cas métier** : Classer les articles candidats d'un utilisateur — les scorer puis publier un top-K — à partir des interactions passées, de la fiche article et du profil utilisateur, sans jamais utiliser d'information postérieure à la session scorée.
**Jeu de données** : `ecommerce_candidate_impressions` — Journal de candidats scorés d'un catalogue e-commerce

> Jeu d'apprentissage **par couple (utilisateur, article candidat)** : chaque ligne décrit un candidat proposé à un utilisateur lors d'une session, avec les features disponibles à cet instant et la pertinence observée ensuite. 1 200 utilisateurs x 30 candidats = 36 000 lignes. La pertinence est construite à partir d'une affinité latente utilisateur-article (goût par catégorie, sensibilité au prix, appétence à la nouveauté) bruitée, de sorte qu'aucun modèle ne puisse atteindre 1,0 : la part irréductible est mesurée et annoncée. Les utilisateurs froids (peu d'historique) et la longue traîne du catalogue sont volontairement surreprésentés par rapport à un journal réel, parce que c'est là que se joue la qualité d'un moteur de recommandation.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Construire un jeu de candidats (utilisateur x article) plutôt qu'une matrice d'interactions, et comprendre ce que ce choix autorise : consommer les features de fiche article, scorer un article neuf, publier sous contraintes métier.
- Formaliser le contrat d'antériorité de chaque feature : connue avant la session (fiche article, historique agrégé), connue au moment de la session (intention récente), ou interdite (issue de la session elle-même).
- Comprendre pourquoi l'unité d'évaluation est l'utilisateur et non la ligne : une précision calculée sur toutes les lignes mêle un client très actif à un nouveau venu et ne décrit aucun des deux.

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (12000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 12000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.45)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `ecommerce_candidate_impressions` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Le jeu est un **journal de candidats scorés**, pas une table d'interactions brutes : l'échantillonnage des négatifs est fait en amont par le générateur (30 candidats par utilisateur, dont ~14 % de pertinents), ce qui est le protocole standard d'entraînement d'un modèle de classement.; Les 30 candidats d'un utilisateur sont tirés à la même date de session : le classement se compare donc à nombre de candidats égal entre utilisateurs, ce qui rend Precision@K et Recall@K comparables. Trente candidats pour un top-10 est un minimum : à 12 candidats le top-10 couvre presque toute la liste, le NDCG sature et même un scoreur aléatoire semble correct (mesuré : NDCG@10 aléatoire de 0,47 à 12 candidats contre 0,23 à 30)..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. La cible de classement et la structure du jeu de candidats

La cible `relevance` vaut 1 quand le couple (utilisateur, article candidat) correspond à
une intention réelle. Mais décrire sa distribution ne dit rien d'utile : ce qui gouverne la
difficulté du problème, c'est la **structure** du jeu — combien de candidats par utilisateur,
combien d'utilisateurs sans aucune intention observable, et si la popularité des articles porte déjà
le signal de pertinence.

Cette dernière question est décisive. Si la popularité ne disait rien de la pertinence, le tri par
popularité — le moteur gratuit que le site utilise aujourd'hui — serait une référence ridicule, et
n'importe quel modèle la battrait sans mérite. Si elle en dit beaucoup, la référence est forte et
le gain du modèle doit se mesurer contre elle.

In [ ]:
# Aucun nom de colonne n'est écrit dans ce notebook : l'évaluateur du projet résout ses réglages
# depuis `conf/config.yaml` (bloc `recommendation`, avec repli sur le noeud `data`). L'EDA décrit
# donc exactement les colonnes que le rapport de production utilisera — renommer un champ dans la
# configuration suffit, aucune édition de notebook n'est nécessaire.
from src.evaluation.evaluator import RankingSettings

SETTINGS = RankingSettings.resolve(CONFIG.model_dump(mode="json"))

TARGET = CONFIG.data.target
GROUP = SETTINGS.group_column
ARTICLE = SETTINGS.item_column
PERIODE = SETTINGS.time_column or CONFIG.data.time_column
print(f"cible                : {TARGET}")
print(f"unité de publication : {GROUP}")
print(f"article              : {ARTICLE}")
print(f"popularité           : {SETTINGS.popularity_column}")
print(f"intention récente    : {SETTINGS.intent_column}")
print(f"disponibilité        : {SETTINGS.stock_column}")
print(
    f"activité             : {SETTINGS.activity_column} (froid <= {SETTINGS.cold_user_max_orders})"
)
print(f"marge                : {SETTINGS.margin_column}")
print(f"coupure publiée      : top-{SETTINGS.top_k}")
print()

prevalence = float(raw[TARGET].mean())
per_user = raw.groupby(GROUP, observed=True)[TARGET].agg(["size", "sum", "mean"])

print(f"couples (utilisateur, candidat) : {len(raw):,}")
print(f"utilisateurs distincts          : {per_user.shape[0]:,}")
print(
    f"candidats par utilisateur       : {per_user['size'].min()} à {per_user['size'].max()}"
    f" (médiane {per_user['size'].median():.0f})"
)
print(f"pertinence globale              : {prevalence:.4f}")
print()
print("Répartition du nombre de candidats pertinents par utilisateur :")
distribution = per_user["sum"].value_counts().sort_index()
display(
    pd.DataFrame(
        {
            "utilisateurs": distribution,
            "part": (distribution / len(per_user)).round(4),
        }
    )
)

sans_intention = int((per_user["sum"] == 0).sum())
print(
    f"utilisateurs sans AUCUN candidat pertinent : {sans_intention}"
    f" ({sans_intention / len(per_user):.1%})"
)
print("Ces utilisateurs sont exclus de toutes les métriques de classement : aucun ordre")
print("ne peut les satisfaire, et les compter comme « parfaits » récompenserait un moteur")
print("qui publie peu.")

**Ce qu'il faut retenir**

- Chaque utilisateur est scoré sur le **même nombre de candidats** : c'est une exigence de comparabilité, sans laquelle Precision@K et Recall@K ne mesureraient pas la même chose d'un utilisateur à l'autre.
- Une part notable d'utilisateurs n'a aucun candidat pertinent. Ils sont exclus des métriques de classement : les inclure reviendrait à récompenser un moteur qui publie peu, puisque publier rien ne peut pas être faux.
- La prévalence globale borne le plancher : un moteur qui tire au hasard obtient une précision égale à cette prévalence. C'est la première valeur à connaître avant de commenter un chiffre.

### 5.1 La popularité porte-t-elle le signal ?

C'est la question qui fixe la difficulté du problème. On regarde la pertinence moyenne par décile
d'audience : si elle croît fortement, le tri par popularité est une référence solide ; si elle est
plate, la référence est vide.

In [ ]:
# La question qui décide de tout : la popularité porte-t-elle un signal de pertinence ?
# Si non, le tri par popularité est une référence ridicule et la comparaison n'apprend rien.
# Si oui — et c'est le cas réel — c'est une référence forte qu'un modèle doit battre nettement.
audience = SETTINGS.popularity_column

# Les colonnes « témoin » du tableau ne sont pas écrites à la main : on prend les trois champs
# numériques les plus corrélés à la cible (hors audience et cible). Le choix est donc dicté par le
# jeu de données, et le tableau reste pertinent si le schéma change.
numeriques = raw.select_dtypes("number").drop(
    columns=[column for column in (TARGET, audience) if column in raw.columns], errors="ignore"
)
# `TARGET` vient d'être écartée de `numeriques` : la corrélation se calcule donc contre la
# colonne brute, pas contre la matrice de corrélation interne.
correlations = numeriques.corrwith(raw[TARGET].astype("float64")).abs().sort_values(ascending=False)
temoins = list(correlations.head(3).index)
print(f"colonnes témoin retenues (|corrélation| à la cible) : {temoins}")
print()

agregats = {
    f"{audience} médian": (audience, "median"),
    "pertinence": (TARGET, "mean"),
    "effectif": (TARGET, "size"),
}
for colonne in temoins:
    agregats[f"{colonne} moyen"] = (colonne, "mean")

lien = (
    raw.groupby(pd.qcut(raw[audience], 10, labels=False, duplicates="drop"), observed=True)
    .agg(**agregats)
    .round(4)
    .rename_axis("décile d'audience")
)
display(lien)

correlation = float(raw[[audience, TARGET]].corr(numeric_only=True).iloc[0, 1])
print(f"corrélation de Pearson {audience} / {TARGET} : {correlation:+.4f}")
print()
print("Lecture : la pertinence croît avec l'audience, mais elle ne s'y réduit pas —")
print("le dixième décile n'est pas pertinent à 100 %. C'est exactement l'espace dans")
print("lequel un modèle doit travailler : capter la part d'audience qui est du signal,")
print("et ajouter l'affinité individuelle que l'audience ne porte pas.")

**Ce qu'il faut retenir**

- La pertinence **croît** avec l'audience : les articles beaucoup vus sont beaucoup vus parce qu'ils plaisent largement. Le tri par popularité n'est donc pas une référence aléatoire, c'est un vrai moteur — gratuit, explicable et robuste.
- Mais elle ne s'y **réduit** pas : le dernier décile n'est pas pertinent à 100 %, et les déciles intermédiaires se chevauchent. C'est dans cet écart que travaille un modèle, en ajoutant l'affinité individuelle que l'audience agrégée ne peut pas porter.
- Cette relation explique aussi le risque de boucle de rétroaction : exposer ce qui est déjà populaire augmente son audience, qui augmente son score, qui augmente son exposition. La mesurer ici, c'est se donner la référence contre laquelle on surveillera la dérive en production.

### 5.2 Ce qui contraint la publication

La pertinence n'est pas la seule contrainte. Un moteur qui publierait uniquement ce qui est
pertinent montrerait des articles en rupture de stock, dix articles de la même catégorie, ou
laisserait les nouveaux utilisateurs sans recommandation exploitable. Ces contraintes se mesurent
avant de modéliser, parce qu'elles déterminent ce que la publication devra filtrer.

In [ ]:
# Ce qui contraint la publication, indépendamment de la pertinence. Chaque indicateur est calculé
# sur une colonne résolue depuis la configuration ; une colonne absente du schéma est écartée au
# lieu de faire échouer le notebook.
def _part(colonne: str, masque) -> tuple[str, float]:
    """Return the indicator name and its share, or NaN when the column is unavailable."""
    if colonne not in raw.columns:
        return colonne, float("nan")
    return colonne, float(masque(raw[colonne]).mean())


lignes: list[dict[str, object]] = []

colonne, valeur = _part(SETTINGS.stock_column, lambda serie: serie <= 0)
lignes.append(
    {
        "indicateur": f"candidats en rupture ({colonne} <= 0)",
        "part": valeur,
        "conséquence sur la publication": "un article indisponible ne doit jamais occuper un "
        "emplacement : le filtre s'applique après le classement",
    }
)

colonne, valeur = _part(SETTINGS.intent_column, lambda serie: serie <= 0)
lignes.append(
    {
        "indicateur": f"candidats sans intention récente ({colonne} <= 0)",
        "part": valeur,
        "conséquence sur la publication": "la référence d'exploitation pure ne peut rien publier "
        "ici : le moteur explore plus qu'il n'exploite",
    }
)

if SETTINGS.activity_column in raw.columns:
    activite = raw.groupby(GROUP, observed=True)[SETTINGS.activity_column].first()
    part_froid = float((activite <= SETTINGS.cold_user_max_orders).mean())
else:
    part_froid = float("nan")
lignes.append(
    {
        "indicateur": f"utilisateurs en démarrage froid ({SETTINGS.activity_column} <= "
        f"{SETTINGS.cold_user_max_orders})",
        "part": part_froid,
        "conséquence sur la publication": "aucun historique exploitable : le service rendu ne peut "
        "reposer que sur la fiche article et le contexte",
    }
)

if SETTINGS.margin_column in raw.columns:
    marge = raw[SETTINGS.margin_column]
    part_marge = float((marge <= marge.median()).mean())
else:
    part_marge = float("nan")
lignes.append(
    {
        "indicateur": f"candidats sous la médiane de marge ({SETTINGS.margin_column})",
        "part": part_marge,
        "conséquence sur la publication": "pertinence et valeur ne coïncident pas : l'arbitrage se "
        "règle dans le score de publication, pas dans le modèle",
    }
)

sans_intention_part = float((per_user["sum"] == 0).mean())
lignes.append(
    {
        "indicateur": f"utilisateurs sans aucun candidat pertinent ({TARGET} toujours 0)",
        "part": sans_intention_part,
        "conséquence sur la publication": "exclus des métriques de classement : aucun ordre "
        "ne peut les satisfaire",
    }
)

contraintes = pd.DataFrame(lignes).set_index("indicateur")
contraintes["part"] = contraintes["part"].astype("float64").round(4)
display(contraintes)

print()
print(f"références du catalogue : {raw[ARTICLE].nunique():,}")
if PERIODE and PERIODE in raw.columns:
    periode = pd.to_datetime(raw[PERIODE])
    print(f"fenêtre observée        : {periode.min().date()} → {periode.max().date()}")
    print(f"nombre de périodes      : {periode.dt.to_period('D').nunique()} jours distincts")

# Cardinalité des champs catégoriels : ce sont les leviers de segmentation du rapport d'erreur.
categorielles = raw.select_dtypes(include=["object", "string", "category", "bool"]).columns
if len(categorielles):
    cardinalites = pd.DataFrame(
        {
            "colonne": list(categorielles),
            "modalités": [int(raw[colonne].nunique()) for colonne in categorielles],
            "part de la modalité majoritaire": [
                round(float(raw[colonne].astype(str).value_counts(normalize=True).iloc[0]), 4)
                for colonne in categorielles
            ],
        }
    ).sort_values("modalités", ascending=False)
    print()
    display(cardinalites)
    print("Une modalité majoritaire à plus de 90 % porte peu d'information ; une cardinalité")
    print("proche du nombre de lignes est un identifiant déguisé, à écarter des features.")

**Ce qu'il faut retenir**

- La majorité des candidats n'a jamais été consultée par l'utilisateur : le moteur est structurellement en **exploration**, pas en exploitation. Un modèle qui ne sait scorer que ce qui a déjà été vu est donc inutile sur la plus grande part du catalogue.
- Les articles en rupture ne doivent jamais occuper un emplacement. Le filtrer dans le modèle est fragile — le stock change toutes les heures, le modèle toutes les semaines — donc le filtre est appliqué après le classement, dans la couche de publication.
- La part d'utilisateurs froids est un **objectif** à part entière, pas une note de bas de page : un moteur excellent pour les clients fidèles et médiocre pour les nouveaux venus est un problème d'acquisition déguisé en problème de précision.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- La pertinence se joue presque entièrement dans les **signaux croisés** : `user_category_affinity`, `price_gap_pct` et `user_item_views_30d` portent l'essentiel du signal, alors que les features purement utilisateur ou purement article ne distinguent pas un bon candidat d'un mauvais.
- `item_views_7d` suit une loi puissance très marquée : classer par cette seule colonne reproduit le moteur historique. C'est une **vraie** baseline — son NDCG@10 de 0,334 bat nettement l'aléatoire (0,230), parce que l'audience observe un attrait latent qui profite aussi à la pertinence — mais elle n'expose que 8,4 % du catalogue dans ses top-10, contre plus de la moitié pour un modèle qui combine attrait et affinité individuelle.
- 72,5 % des candidats n'ont jamais été consultés par l'utilisateur (`user_item_views_30d = 0`) et 62,1 % n'ont jamais été vus du tout (`days_since_last_view` manquant) : le moteur travaille majoritairement en exploration, pas en exploitation. Le signal d'intention est rare mais très prédictif quand il existe.
- Les utilisateurs froids (au plus 1 commande sur 12 mois) représentent ~29 % des lignes mais une part bien plus faible des positifs : sans traitement dédié, un modèle global les classe mal et le moteur perd précisément les clients à convertir.
- `item_stock_units = 0` sur ~7 % des candidats, et davantage sur les articles les plus populaires : filtrer après scoring coûte donc de la pertinence mesurée, ce qui doit être chiffré plutôt que supposé nul.
- `price_gap_pct` a un effet en cloche inversée : un candidat très en dessous du prix habituel n'est pas mieux classé qu'un candidat aligné, ce qu'un modèle linéaire ne capture pas et qu'un arbre capture sans feature supplémentaire.

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).